# ANN-SNN Scaling Laws — Full Sweep on Colab GPU

**Runtime:** ~1-2 hours on T4 GPU

Make sure you set: Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Step 1: Clone and setup
!rm -rf ann_snn_scaling_laws
!git clone https://github.com/Vighneshwarkuru/ann_snn_scaling_laws.git
%cd ann_snn_scaling_laws
!pip install -q torch torchvision spikingjelly scipy numpy pandas matplotlib seaborn pyyaml ptflops tqdm psutil

In [ ]:
# Step 2: Verify setup
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - SET T4 GPU!'}")
print(f"PyTorch: {torch.__version__}")

import sys
sys.path.insert(0, '.')
from src.models import get_model
from src.data.datasets import get_dataloaders, get_dataset_info
from src.training.train_ann import load_or_train_ann
from src.conversion.ann_to_snn import ANNtoSNNConverter
from src.evaluation.evaluate_ann import evaluate_ann
from src.evaluation.evaluate_snn import evaluate_snn
print("All imports OK!")

In [ ]:
# Step 3: Clean old results and run the full experiment
import shutil, os
for d in ['results_real/raw', 'checkpoints']:
    if os.path.exists(d):
        shutil.rmtree(d)
os.makedirs('results_real/raw', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print("Clean slate ready.")

In [ ]:
# Step 4: Run the experiment (uses run_real_experiment.py with all fixes)
# This does: 3 datasets x 2 models x 6 timesteps = 36 experiments
!python scripts/run_real_experiment.py

In [ ]:
# Step 5: Check results
import glob
csvs = sorted(glob.glob('results_real/raw/*.csv'))
print(f"Completed: {len(csvs)} experiments\n")
for f in csvs:
    name = os.path.basename(f).replace('.csv','')
    with open(f) as fh:
        lines = fh.readlines()
        if len(lines) >= 2:
            acc = lines[1].split(',')[6]
            print(f"  {name}: acc={float(acc)*100:.2f}%")

In [ ]:
# Step 6: Run analysis
!python scripts/analyze_results.py --results-dir results_real

In [ ]:
# Step 7: Generate plots
!python scripts/generate_plots.py --results-dir results_real

In [ ]:
# Step 8: Display key results
import pandas as pd
from pathlib import Path

csvs = list(Path('results_real/raw').glob('*.csv'))
df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
print(f"Total experiments: {len(df)}")
print(f"Models: {sorted(df['model'].unique())}")
print(f"Datasets: {sorted(df['dataset'].unique())}")
print(f"Timesteps: {sorted(df['T'].unique())}")

print("\n--- Accuracy Table ---")
pivot = df.pivot_table(values='snn_accuracy', index=['model','dataset'], columns='T', aggfunc='mean')
print((pivot * 100).round(2).to_string())

In [ ]:
# Step 9: Plot scaling curves
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, dataset in enumerate(sorted(df['dataset'].unique())[:3]):
    ax = axes[i]
    ax.set_title(dataset.upper())
    for model in sorted(df['model'].unique()):
        sub = df[(df['dataset']==dataset) & (df['model']==model)].sort_values('T')
        if len(sub) >= 3:
            ax.plot(sub['T'], sub['snn_accuracy']*100, 'o-', label=model, markersize=6)
    ax.set_xlabel('Timesteps (T)')
    ax.set_ylabel('SNN Accuracy (%)')
    ax.set_xscale('log', base=2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 105)

plt.suptitle('Scaling Laws: Accuracy vs Timesteps (Real Data)', fontweight='bold')
plt.tight_layout()
plt.savefig('results_real/scaling_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 10: Download results
!zip -r /content/real_results.zip results_real/ checkpoints/ paper/figures/
from google.colab import files
files.download('/content/real_results.zip')